<a href="https://colab.research.google.com/github/Givi-Modebadze/ML_Final_Project/blob/main/experiments/Arima_Regressor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

FOLDERNAME = 'ML_Final_Project/walmart-recruiting-store-sales-forecasting'
assert FOLDERNAME is not None, "[!] Enter the foldername."

import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

%cd /content/drive/My\ Drive/$FOLDERNAME/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/My Drive/ML_Final_Project/walmart-recruiting-store-sales-forecasting


In [2]:
import torch # Main PyTorch Library
from torch import nn # Used for creating the layers and loss function
from torch.optim import Adam # Adam Optimizer
import torchvision.transforms as transforms # Transform function used to modify and preprocess all the images
from torch.utils.data import Dataset, DataLoader # Dataset class and DataLoader for creating the objects
from sklearn.preprocessing import LabelEncoder # Label Encoder to encode the classes from strings to numbers
import matplotlib.pyplot as plt # Used for visualizing the images and plotting the training progress
from PIL import Image # Used to read the images from the directory
import pandas as pd # Used to read/create dataframes (csv) and process tabular data
import numpy as np # preprocessing and numerical/mathematical operations
import os # Used to read the images path from the directory

device = "cuda" if torch.cuda.is_available() else "cpu" # detect the GPU if any, if not use CPU, change cuda to mps if you have a mac
print("Device available: ", device)

Device available:  cuda


# Merge data

In [3]:
df = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
stores = pd.read_csv('stores.csv')
features = pd.read_csv('features.csv')

def merger(df, stores, features):
    df = df.merge(stores, how='left', on='Store')
    df = df.merge(features, how='left', on=['Store', 'Date', 'IsHoliday'])
    return df

df = merger(df, stores, features)
test = merger(test, stores, features)

df.sort_values('Date', inplace=True)
test.sort_values('Date', inplace=True)

y = df['Weekly_Sales']
X = df.drop(columns=['Weekly_Sales'])
dates = df['Date']

# print(df.head())
# print(test.head())

# Preprocessor

In [4]:
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

class Preprocessor(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.type_encoder = None

    def fit(self, X: pd.DataFrame, y=None):
        if 'Type' in X.columns:
            self.type_encoder = X['Type'].astype('category').cat.categories
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        X = X.copy()

        X['Date'] = pd.to_datetime(X['Date'])
        X['Year'] = X['Date'].dt.year
        X['Month'] = X['Date'].dt.month
        X['Day'] = X['Date'].dt.day

        if self.type_encoder is not None:
            X['Type'] = pd.Categorical(X['Type'], categories=self.type_encoder)
            X['Type'] = X['Type'].cat.codes

        if 'IsHoliday' in X.columns:
            X['IsHoliday'] = X['IsHoliday'].astype(int)

        return X


In [5]:
from sklearn.base import BaseEstimator, TransformerMixin
import pandas as pd

class FeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.holidays = {
            'IsSuperBowl': ['2010-02-12', '2011-02-11', '2012-02-10', '2013-02-08'],
            'IsLaborDay': ['2010-09-10', '2011-09-09', '2012-09-07', '2013-09-06'],
            'IsThanksgiving': ['2010-11-26', '2011-11-25', '2012-11-23', '2013-11-29'],
            'IsChristmas': ['2010-12-31', '2011-12-30', '2012-12-28', '2013-12-27']
        }

    def fit(self, X, y=None):
        X = X.copy()
        X['Date'] = pd.to_datetime(X['Date'])
        X['Weekly_Sales'] = y.values
        X['Year'] = X['Date'].dt.year
        X['Week'] = X['Date'].dt.isocalendar().week.astype(int)
        self.full_data_ = X[['Store', 'Dept', 'Date', 'Year', 'Week', 'Weekly_Sales']].copy()
        return self

    def transform(self, X):
        X = X.copy()
        X['Date'] = pd.to_datetime(X['Date'])
        X['Year'] = X['Date'].dt.year
        X['Month'] = X['Date'].dt.month
        X['Week'] = X['Date'].dt.isocalendar().week.astype(int)
        X['Day'] = X['Date'].dt.day
        X['DayOfWeek'] = X['Date'].dt.dayofweek

        for k in range(1, 3):
            X[f'Week_sin_{k}'] = np.sin(2 * np.pi * k * X['Week'] / 52)
            X[f'Week_cos_{k}'] = np.cos(2 * np.pi * k * X['Week'] / 52)

        X['DayOfYear'] = X['Date'].dt.dayofyear
        for k in range(1, 3):
            X[f'DayOfYear_sin_{k}'] = np.sin(2 * np.pi * k * X['DayOfYear'] / 365)
            X[f'DayOfYear_cos_{k}'] = np.cos(2 * np.pi * k * X['DayOfYear'] / 365)


        for holiday_name, date_list in self.holidays.items():
            holiday_dates = pd.to_datetime(date_list)

            X[holiday_name] = X['Date'].isin(holiday_dates).astype(int)
            week_before = holiday_dates - pd.Timedelta(weeks=1)
            week_after = holiday_dates + pd.Timedelta(weeks=1)

            X[f'{holiday_name}Before'] = X['Date'].isin(week_before).astype(int)
            X[f'{holiday_name}After'] = X['Date'].isin(week_after).astype(int)

        base = self.full_data_.copy()
        last_year = base.copy()
        last_year['Year'] += 1
        last_year = last_year.rename(columns={'Weekly_Sales': 'Sales_LastYear'})

        df = X.merge(
            last_year[['Store', 'Dept', 'Year', 'Week', 'Sales_LastYear']],
            on=['Store', 'Dept', 'Year', 'Week'],
            how='left'
        )

        two_years_ago = base.copy()
        two_years_ago['Year'] += 2
        two_years_ago = two_years_ago.rename(columns={'Weekly_Sales': 'Sales_TwoYearsAgo'})

        df = df.merge(
            two_years_ago[['Store', 'Dept', 'Year', 'Week', 'Sales_TwoYearsAgo']],
            on=['Store', 'Dept', 'Year', 'Week'],
            how='left'
        )

        df['PrevYearSales'] = df['Sales_LastYear'].fillna(df['Sales_TwoYearsAgo'])
        df.drop(columns=['Sales_LastYear', 'Sales_TwoYearsAgo'], inplace=True)

        return df

In [6]:
from sklearn.base import BaseEstimator, TransformerMixin

class FeatureSelector(BaseEstimator, TransformerMixin):
    def __init__(self, columns_to_drop=None):
        self.columns_to_drop = columns_to_drop or []

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        return X.drop(columns=self.columns_to_drop, errors='ignore')

# cross validation

In [7]:
def weighted_mae(y_true, y_pred, is_holiday):
    weights = is_holiday * 4 + 1
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

In [8]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error

def cross_validation(pipeline, X, y, df_with_dates):
    from matplotlib.dates import DateFormatter

    tscv = TimeSeriesSplit(n_splits=6)
    rmse_scores = []
    wmae_scores = []
    train_rmse_scores = []
    train_wmae_scores = []

    for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
        if fold < 4:
          continue

        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        pipeline.fit(X_train, y_train)

        y_val_pred = pipeline.predict(X_val)
        y_train_pred = pipeline.predict(X_train)

        val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
        train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))

        val_is_holiday = df_with_dates.iloc[val_idx]['IsHoliday'].astype(int)
        train_is_holiday = df_with_dates.iloc[train_idx]['IsHoliday'].astype(int)

        val_wmae = weighted_mae(y_val, y_val_pred, val_is_holiday)
        train_wmae = weighted_mae(y_train, y_train_pred, train_is_holiday)

        print(f"\nFold {fold + 1} Scores:")
        print(f"Train RMSE: {train_rmse:.2f} | Train WMAE: {train_wmae:.2f}")
        print(f"Valid RMSE: {val_rmse:.2f} | Valid WMAE: {val_wmae:.2f}")

        rmse_scores.append(val_rmse)
        wmae_scores.append(val_wmae)
        train_rmse_scores.append(train_rmse)
        train_wmae_scores.append(train_wmae)

        df_all = pd.DataFrame({
            'Date': pd.to_datetime(df_with_dates['Date']),
            'y': y.values
        })
        df_all = df_all.groupby('Date').sum().reset_index().sort_values('Date')

        df_val = pd.DataFrame({
            'Date': pd.to_datetime(df_with_dates.iloc[val_idx]['Date']),
            'Actual': y_val.values,
            'Predicted': y_val_pred
        }).groupby('Date').sum().reset_index().sort_values('Date')

        df_train = pd.DataFrame({
            'Date': pd.to_datetime(df_with_dates.iloc[train_idx]['Date']),
            'Actual': y_train.values,
            'Predicted': y_train_pred
        }).groupby('Date').sum().reset_index().sort_values('Date')

        plt.figure(figsize=(12, 4))

        plt.plot(df_all['Date'], df_all['y'], color='gray', label='Train Actuals', linewidth=1.5, alpha=0.4)

        plt.plot(df_train['Date'], df_train['Predicted'], color='green', label='Train Predicted', linewidth=1.5, alpha=0.8)

        plt.plot(df_val['Date'], df_val['Actual'], color='blue', label='Val Actuals', linewidth=2)

        plt.plot(df_val['Date'], df_val['Predicted'], color='orange', label='Val Predicted', linewidth=2)

        plt.title(f'Fold {fold + 1} - Weekly Sales Over Time', fontsize=14)
        plt.xlabel("Date")
        plt.ylabel("Weekly Sales")
        plt.grid(True, linestyle='--', alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.show()

    print(f"\nMean Train RMSE: {np.mean(train_rmse_scores):.2f}")
    print(f"Mean Train WMAE: {np.mean(train_wmae_scores):.2f}")
    print(f"Mean Valid RMSE: {np.mean(rmse_scores):.2f}")
    print(f"Mean Valid WMAE: {np.mean(wmae_scores):.2f}")


# Pipeline

In [16]:
from sklearn.base import BaseEstimator, RegressorMixin
from statsmodels.tsa.arima.model import ARIMA
import warnings
warnings.filterwarnings('ignore')

class ARIMARegressor(BaseEstimator, RegressorMixin):
    def __init__(self, order=(1, 1, 1)):
        self.order = order
        self.models = {}
        self.mean_value = None
        self.store_means = {}
        self.avg_dept_per_store = None

    def fit(self, X, y):
        self.mean_value = np.mean(y)

        df_temp = X.copy()
        df_temp['y'] = y
        df_temp['Date'] = pd.to_datetime(df_temp['Date'])
        df_temp = df_temp.sort_values('Date')

        self.avg_dept_per_store = df_temp.groupby('Store')['Dept'].nunique().mean()

        for store, group in df_temp.groupby(['Store']):
            key = f"store_{store}"

            ts = group.groupby('Date')['y'].sum().sort_index()
            self.store_means[key] = ts.mean()

            try:
                if len(ts) >= 10:
                    model = ARIMA(ts, order=self.order)
                    fitted_model = model.fit()
                    self.models[key] = fitted_model
            except:
                pass

        return self

    def predict(self, X):
        predictions = []

        for idx, row in X.iterrows():
            store_key = f"store_{row['Store']}"

            if store_key in self.models:
                try:
                    forecast = self.models[store_key].forecast(steps=1)
                    store_forecast = forecast.iloc[0] if hasattr(forecast, 'iloc') else forecast[0]

                    prediction = store_forecast / self.avg_dept_per_store
                    predictions.append(max(0, prediction))
                except:
                    predictions.append(self.store_means.get(store_key, self.mean_value))
            else:
                predictions.append(self.mean_value)

        return np.array(predictions)

In [ ]:
from sklearn.pipeline import Pipeline

preprocessor = Preprocessor()
feature_selection = FeatureSelector(columns_to_drop=['IsHoliday'])
feature_engineer = FeatureEngineer()
model = ARIMARegressor()

pipeline = Pipeline([
    ('feature_engineer', feature_engineer),
    ('preprocessor', preprocessor),
    ('feature selection', feature_selection),
    ('model', model)
])
cross_validation(pipeline, X, y, df)

In [ ]:
pipeline.fit(X, y)

Pipeline(steps=[('preprocessor', Preprocessor()), ('model', ARIMARegressor())])